In [1]:
# %pip install langchain-experimental
# %pip install open_clip_torch

In [2]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_experimental.open_clip import OpenCLIPEmbeddings
import glob
import base64

paths = glob.glob('../images/*.jpeg', recursive=True)

In [3]:
embedding_model = OpenCLIPEmbeddings() #(model="ViT-H-14", device="cuda")

In [4]:
embedings = embedding_model.embed_image(paths)

In [5]:
embedings[0]

[-0.05834139510989189,
 -0.029942527413368225,
 0.0038106299471110106,
 -0.01514993142336607,
 -0.0449778214097023,
 0.019940536469221115,
 -0.0048165088519454,
 -0.059451665729284286,
 0.01590549945831299,
 -0.015838058665394783,
 0.00358516164124012,
 0.006735948845744133,
 0.005546157248318195,
 -0.023108989000320435,
 -0.0332687608897686,
 -0.058386363089084625,
 0.00013643171405419707,
 -0.056548163294792175,
 -0.023857925087213516,
 -0.06229807063937187,
 -0.030631545931100845,
 -0.0036633035633713007,
 0.02422552928328514,
 0.0631798654794693,
 -0.03214956820011139,
 0.010108794085681438,
 0.02757791243493557,
 0.03919951617717743,
 0.034878406673669815,
 0.035580188035964966,
 -0.04691539332270622,
 -0.006264159921556711,
 -0.005141493398696184,
 -0.05407610535621643,
 0.027428967878222466,
 -0.0038469659630209208,
 -0.0073664081282913685,
 0.006542072165757418,
 0.020200831815600395,
 0.0715680867433548,
 -0.008402110077440739,
 -0.006548732984811068,
 -0.022583242505788803,
 

In [6]:
lc_docs = []
def encode_image(path):
    with open(path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

for path in paths:
    doc = Document(
        page_content=encode_image(path),
        metadata ={
            'source': path
        }
    )

    lc_docs.append(doc)

In [7]:
vector_store = FAISS.from_documents(lc_docs, embedding=OpenCLIPEmbeddings())

In [8]:
retriever = vector_store.as_retriever()

In [9]:
dog_paths = glob.glob('../images/dog*.jpeg', recursive=True)

In [10]:
dog_to_cat = {}
for dog_pic in dog_paths:
    docs = retriever.invoke(encode_image(dog_pic))
    cats_retrieved = 0
    for i, doc in enumerate(docs):
        if "cat" in doc.metadata["source"]:            
            cats_retrieved += 4 - i
    dog_to_cat[dog_pic] = cats_retrieved

In [11]:
dog_to_cat

{'../images/dog_1.jpeg': 5,
 '../images/dog_2.jpeg': 2,
 '../images/dog_3.jpeg': 3,
 '../images/dog_4.jpeg': 1,
 '../images/dog_5.jpeg': 1}

In [12]:
for i, doc in enumerate(docs):
    print(f"Rank {i+1}: {doc.metadata['source']}")

Rank 1: ../images/dog_5.jpeg
Rank 2: ../images/dog_3.jpeg
Rank 3: ../images/dog_2.jpeg
Rank 4: ../images/cat_5.jpeg
